# Neural Network Classification with TensorFlow/Keras

This notebook covers:
- Binary classification (2 classes) with circles dataset
- Multi-class classification (10 classes) with Fashion-MNIST
- Activation functions: ReLU and Sigmoid
- Visualizing decision boundaries
- Learning rate scheduling and optimization
- Evaluating models with confusion matrices
- Data normalization for better performance

## Setup and Imports

In [ ]:
# Import TensorFlow
import tensorflow as tf

# Check version (should be 2.x+)
print(tf.__version__)

# Print timestamp for tracking notebook runs
import datetime
print(f"Notebook last run (end-to-end): {datetime.datetime.now()}")

In [ ]:
# Import supporting libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Check NumPy version
print(f"NumPy: {np.__version__}")

## Part 1: Binary Classification - Circles Dataset

A classic non-linear classification problem: separating two concentric circles.

In [ ]:
# Import dataset generator from scikit-learn
from sklearn.datasets import make_circles

In [ ]:
# Create circles dataset
n_samples = 1000  # Total number of points

# Generate data
# Two concentric circles: inner circle (class 0), outer circle (class 1)
X, y = make_circles(
    n_samples=n_samples,
    noise=0.05,  # Small amount of randomness
    random_state=datetime.datetime.now().microsecond  # Random seed for variability
)

In [ ]:
# Create DataFrame to inspect data
# X0 and X1 are the two features (x and y coordinates)
circles = pd.DataFrame({"X0": X[:, 0], "X1": X[:, 1], "label": y})
circles.info()

In [ ]:
# Check class distribution
# Should be roughly balanced (500 samples per class)
circles.label.value_counts()

In [ ]:
# Visualize the circles dataset
# Red/yellow colors represent different classes
# Notice: cannot be separated by a straight line (non-linear problem)
plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu)
plt.show()

## Building Model 1: Without Activation Functions

In [ ]:
# Set random seed for reproducibility
tf.random.set_seed(datetime.datetime.now().microsecond)

# Build neural network (3 layers, no activation functions)
# This won't work well for non-linear data!
model_1 = tf.keras.Sequential([
    tf.keras.layers.Dense(100),  # 100 neurons in first layer
    tf.keras.layers.Dense(10),   # 10 neurons in second layer
    tf.keras.layers.Dense(1)     # 1 neuron for binary output
])

# Compile model
model_1.compile(
    # Binary Crossentropy: standard loss for binary classification
    # Measures how far predictions are from actual labels
    loss=tf.keras.losses.BinaryCrossentropy(),
    
    # Adam optimizer: adaptive learning rate
    optimizer=tf.keras.optimizers.Adam(),
    
    # Track accuracy during training
    metrics=['accuracy']
)

# Train the model
# verbose=2: print one line per epoch
model_1.fit(X, y, epochs=100, verbose=2)

## Visualizing Decision Boundaries

In [ ]:
# Function to visualize model's decision boundary
def plot_decision_boundary(model, X, y):
    """
    Plots the decision boundary created by a classification model.
    
    Shows where the model draws the line (or curve) between classes.
    Adapted from CS231n and Made with ML.
    
    Args:
        model: Trained Keras model
        X: Feature data (shape: n_samples, 2)
        y: Labels (shape: n_samples,)
    """
    # Define plot boundaries (slightly beyond data range)
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    
    # Create a grid of points to predict on
    # meshgrid creates a 2D grid covering the entire plot area
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100)
    )

    # Stack grid coordinates into format model expects
    # np.c_ stacks arrays column-wise
    x_in = np.c_[xx.ravel(), yy.ravel()]

    # Make predictions on all grid points
    y_pred = model.predict(x_in)

    # Check if multi-class or binary
    if model.output_shape[-1] > 1:
        # Multi-class: get class with highest probability
        print("doing multiclass classification...")
        y_pred = np.argmax(y_pred, axis=1).reshape(xx.shape)
    else:
        # Binary: round to 0 or 1
        print("doing binary classification...")
        y_pred = np.round(np.max(y_pred, axis=1)).reshape(xx.shape)

    # Plot the decision boundary as a filled contour
    # Color regions show where model predicts each class
    plt.contourf(xx, yy, y_pred, cmap=plt.cm.RdYlBu, alpha=0.7)
    
    # Overlay actual data points
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.RdYlBu)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())

In [ ]:
# Visualize model_1's decision boundary
# Without activation functions, can only create linear boundaries
# This won't fit the circular data well!
plot_decision_boundary(model_1, X, y)

## Model 2: Adding Activation Functions

ReLU and Sigmoid activations allow the network to learn non-linear patterns.

In [ ]:
# Build improved model with activation functions
tf.random.set_seed(datetime.datetime.now().microsecond)

model_2 = tf.keras.Sequential([
    # ReLU (Rectified Linear Unit): max(0, x)
    # Introduces non-linearity, allowing curved decision boundaries
    tf.keras.layers.Dense(5, activation=tf.keras.activations.relu),
    tf.keras.layers.Dense(5, activation=tf.keras.activations.relu),
    tf.keras.layers.Dense(5, activation=tf.keras.activations.relu),
    
    # Sigmoid: squashes output to (0, 1) range
    # Perfect for binary classification probabilities
    tf.keras.layers.Dense(1, activation=tf.keras.activations.sigmoid)
])

model_2.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

# Train with fewer epochs (model learns faster with activations)
model_2.fit(X, y, epochs=50, verbose=2)

In [ ]:
# View improved decision boundary
# Should now show a curved boundary that separates circles!
plot_decision_boundary(model_2, X, y)

## Creating Train/Test Split

In [ ]:
# Split data: 80% train, 20% test
split = int(len(X) * 0.2)

# Training set (last 80%)
X_train, y_train = X[split:], y[split:]

# Test set (first 20%)
X_test, y_test = X[:split], y[:split]

In [ ]:
# Check training set size
len(X_train)

## Model 3: Training on Split Data

In [ ]:
# Build model_3 (smaller than model_2)
tf.random.set_seed(datetime.datetime.now().microsecond)

model_3 = tf.keras.Sequential([
    # Fewer neurons (4 instead of 5)
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    # Sigmoid output for binary classification
    tf.keras.layers.Dense(1, activation=tf.keras.activations.sigmoid)
])

model_3.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    # Specify learning rate explicitly
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

# Train on training set only
# Save history for plotting loss curves
history = model_3.fit(X_train, y_train, epochs=50, verbose=2)

In [ ]:
# Evaluate on test set
# Returns [loss, accuracy]
loss, accuracy = model_3.evaluate(X_test, y_test)
print(f"Model loss on the test set: {loss}")
print(f"Model accuracy on the test set: {100*accuracy:.2f}%")

In [ ]:
# Plot decision boundaries for train and test sets
# Both should look similar if model generalizes well
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_3, X=X_train, y=y_train)

plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_3, X=X_test, y=y_test)

plt.show()

In [ ]:
# Plot training curves (loss and accuracy over epochs)
# Shows how well the model learned during training
df = pd.DataFrame(history.history)

df.plot()
plt.title("Model_3 training curves")
# Reference lines at 0 and 1
plt.axhline(y=0.0, c="k", ls="--", lw=0.5)
plt.axhline(y=1.0, c="k", ls="--", lw=0.5)
plt.ylim(-0.1, 1.1)
plt.show()

## Part 2: Learning Rate Scheduling

Finding the optimal learning rate can significantly improve training.

In [ ]:
# Build model_4 (same architecture as model_3)
tf.random.set_seed(datetime.datetime.now().microsecond)

model_4 = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation=tf.keras.activations.sigmoid)
])

# Can use string shortcuts for loss and optimizer
model_4.compile(
    loss="binary_crossentropy",  # Same as tf.keras.losses.BinaryCrossentropy()
    optimizer="Adam",  # Uses default Adam settings
    metrics=["accuracy"]
)

# Create learning rate scheduler callback
# Gradually increases learning rate to find optimal value
# Formula: lr = 1e-4 * 10^(epoch/20)
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(
    lambda epoch: 1e-4 * 10**(epoch/20)
)

# Train with learning rate schedule
history = model_4.fit(X_train, y_train, epochs=100, callbacks=[lr_scheduler])

In [ ]:
# Plot training curves for model_4
df = pd.DataFrame(history.history)

df.plot()
plt.title("Model_4 training curves")
plt.axhline(y=0.0, c="k", ls="--", lw=0.5)
plt.axhline(y=1.0, c="k", ls="--", lw=0.5)
plt.ylim(-0.1, 1.1)
plt.show()

In [ ]:
# Calculate learning rates used during training
lr_li = 1e-4 * 10**(np.arange(100)/20)

# Plot learning rate vs loss
# The "valley" shows the optimal learning rate range
plt.figure(figsize=(10, 7))
plt.semilogx(lr_li, history.history["loss"])
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.title("Learning rate vs. loss")
plt.show()

## Model 5: Using Optimal Learning Rate

In [ ]:
# Train model with learning rate from the valley of the curve
# Around 0.02 appears to be optimal from the plot
tf.random.set_seed(datetime.datetime.now().microsecond)

model_5 = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation=tf.keras.activations.sigmoid)
])

model_5.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    # Use optimal learning rate from experiment
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.02),
    metrics=['accuracy']
)

# Train with optimal learning rate
history = model_5.fit(X_train, y_train, epochs=50, verbose=2)

In [ ]:
# Evaluate model_5 on test set
# Should perform better with optimized learning rate
model_5.evaluate(X_test, y_test)

In [ ]:
# Plot decision boundaries for model_5
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_5, X=X_train, y=y_train)

plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_5, X=X_test, y=y_test)

plt.show()

## Evaluating with Confusion Matrix

In [ ]:
# Import confusion matrix from scikit-learn
from sklearn.metrics import confusion_matrix

In [ ]:
# Make predictions on test set
y_preds = model_5.predict(X_test)

# Round probabilities to binary predictions (0 or 1)
y_preds = tf.round(y_preds)

# Create confusion matrix
# Shows: [True Negatives, False Positives]
#        [False Negatives, True Positives]
confusion_matrix(y_test, y_preds)

## Pretty Confusion Matrix Visualization

In [ ]:
# Function to create a labeled confusion matrix
# Adapted from Scikit-Learn and Made with ML
import itertools

figsize = (10, 10)

# Create confusion matrix
cm = confusion_matrix(y_test, tf.round(y_preds))

# Normalize: show percentages instead of counts
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
n_classes = cm.shape[0]

# Create plot
fig, ax = plt.subplots(figsize=figsize)
# Color intensity shows prediction accuracy (darker = better)
cax = ax.matshow(cm, cmap=plt.cm.Blues)
fig.colorbar(cax)

# Set up class labels (0 and 1 for binary)
classes = False
if classes:
    labels = classes
else:
    labels = np.arange(cm.shape[0])

# Label axes
ax.set(
    title="Confusion Matrix",
    xlabel="Predicted label",
    ylabel="True label",
    xticks=np.arange(n_classes),
    yticks=np.arange(n_classes),
    xticklabels=labels,
    yticklabels=labels
)

# Move x-axis labels to bottom
ax.xaxis.set_label_position("bottom")
ax.xaxis.tick_bottom()

# Adjust font sizes
ax.xaxis.label.set_size(20)
ax.yaxis.label.set_size(20)
ax.title.set_size(20)

# Color threshold for text visibility
threshold = (cm.max() + cm.min()) / 2.

# Add text showing counts and percentages
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(
        j, i, f"{cm[i, j]} ({cm_norm[i, j]*100:.1f}%)",
        horizontalalignment="center",
        color="white" if cm[i, j] > threshold else "black",
        size=15
    )

## Part 3: Multi-class Classification - Fashion-MNIST

Classifying clothing items into 10 categories.

In [ ]:
# Import Fashion-MNIST dataset (built into Keras)
from tensorflow.keras.datasets import fashion_mnist

In [ ]:
# Load data (already split into train and test)
# 60,000 training images, 10,000 test images
# Each image is 28x28 pixels, grayscale
(train_data, train_labels), (test_data, test_labels) = fashion_mnist.load_data()

In [ ]:
# Visualize a single example
# Images are stored as 2D arrays (28x28)
import matplotlib.pyplot as plt
plt.imshow(train_data[15]);

In [ ]:
# Define class names
# Labels are integers 0-9, we map them to descriptive names
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Number of output neurons needed (one per class)
len(class_names)

In [ ]:
# Plot multiple random examples
import random

plt.figure(figsize=(7, 7))
for i in range(4):
    ax = plt.subplot(2, 2, i + 1)
    rand_index = random.choice(range(len(train_data)))
    # Display image in grayscale
    plt.imshow(train_data[rand_index], cmap=plt.cm.binary)
    plt.title(class_names[train_labels[rand_index]])
    plt.axis(False)

## Model 6: First Fashion-MNIST Model (Without Normalization)

In [ ]:
# Set random seed
SEED = datetime.datetime.now().microsecond
tf.random.set_seed(SEED)

# Build multi-class classification model
model_6 = tf.keras.Sequential([
    # Flatten: convert 28x28 image to 784-element vector
    # input_shape=(28,28) specifies expected image dimensions
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    
    # Hidden layers with ReLU activation
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    
    # Output layer: 10 neurons (one per class)
    # Softmax: converts raw outputs to probabilities that sum to 1
    tf.keras.layers.Dense(10, activation="softmax")
])

# Compile model
model_6.compile(
    # SparseCategoricalCrossentropy: for multi-class with integer labels
    # "Sparse" means labels are integers (0-9), not one-hot encoded
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Train model on non-normalized data
# Pixel values are currently 0-255
non_norm_history = model_6.fit(
    train_data,
    train_labels,
    epochs=10
)

In [ ]:
# Display model architecture
# Note: "None" in (None, 784) is batch_size (varies during training)
model_6.summary()

## Improving Performance with Normalization

In [ ]:
# Normalize pixel values to [0, 1] range
# Original values are 0-255 (uint8)
# Dividing by 255 scales to 0.0-1.0 (float)
# This helps neural networks learn more efficiently!
train_data = train_data / 255.0
test_data = test_data / 255.0

In [ ]:
# Build model_7 (same architecture as model_6)
tf.random.set_seed(SEED)

model_7 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model_7.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Train on normalized data
# Should learn faster and better than model_6
norm_history = model_7.fit(
    train_data,
    train_labels,
    epochs=10
)

In [ ]:
# Compare training curves: normalized vs non-normalized
# Normalized data should show better/faster learning

# Non-normalized
pd.DataFrame(non_norm_history.history).plot(title="Non-normalized Data")

# Normalized
pd.DataFrame(norm_history.history).plot(title="Normalized data");

## Creating Reusable Confusion Matrix Function

In [ ]:
# Reusable function for creating pretty confusion matrices
import itertools
from sklearn.metrics import confusion_matrix

def make_confusion_matrix(y_true, y_pred, classes=None, figsize=(10, 10), text_size=15):
    """
    Creates a labeled confusion matrix comparing predictions to ground truth.

    Args:
        y_true: Array of true labels
        y_pred: Array of predicted labels
        classes: List of class names (optional)
        figsize: Figure size (default: 10x10)
        text_size: Text size for labels (default: 15)

    Returns:
        Confusion matrix plot
    """
    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Normalize to get percentages
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    n_classes = cm.shape[0]

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    # Darker blue = better prediction accuracy
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(cax, shrink=0.5)

    # Set up labels
    if classes:
        labels = classes
    else:
        labels = np.arange(cm.shape[0])

    # Label axes
    ax.set(
        title="Confusion Matrix",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=np.arange(n_classes),
        yticks=np.arange(n_classes),
        xticklabels=labels,
        yticklabels=labels
    )

    # Move x-labels to bottom
    ax.xaxis.set_label_position("bottom")
    ax.xaxis.tick_bottom()

    # Threshold for text color
    threshold = (cm.max() + cm.min()) / 2.

    # Add text to each cell
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(
            j, i, f"{cm[i, j]} ({cm_norm[i, j]*100:.1f}%)",
            horizontalalignment="center",
            color="white" if cm[i, j] > threshold else "black",
            size=text_size
        )

In [ ]:
# Make predictions on test set
y_probs = model_7.predict(test_data)  # Get probabilities for each class

# Convert probabilities to class predictions
# argmax returns index of highest probability (the predicted class)
y_preds = y_probs.argmax(axis=1)

In [ ]:
# Create confusion matrix with class names
# Diagonal = correct predictions
# Off-diagonal = misclassifications (shows which classes get confused)
make_confusion_matrix(
    y_true=test_labels,
    y_pred=y_preds,
    classes=class_names,
    figsize=(15, 15),
    text_size=10
)

## Visualizing Individual Predictions

In [ ]:
# Function to plot random predictions
import random

def plot_random_image(model, images, true_labels, classes):
    """
    Plots a random image with its prediction and true label.

    Args:
        model: Trained model
        images: Array of images
        true_labels: Array of true labels
        classes: List of class names

    Returns:
        Plot showing image with prediction (green if correct, red if wrong)
    """
    # Pick random image
    i = random.randint(0, len(images))

    # Get predictions
    target_image = images[i]
    # Reshape for model: (28, 28) -> (1, 28, 28)
    pred_probs = model.predict(target_image.reshape(1, 28, 28))
    pred_label = classes[pred_probs.argmax()]
    true_label = classes[true_labels[i]]

    # Plot image
    plt.imshow(target_image, cmap=plt.cm.binary)

    # Color title based on correctness
    if pred_label == true_label:
        color = "green"
    else:
        color = "red"

    # Add prediction info to label
    plt.xlabel(
        "Pred: {} {:2.0f}% (True: {})".format(
            pred_label,
            100 * tf.reduce_max(pred_probs),
            true_label
        ),
        color=color
    )

In [ ]:
# Visualize a random prediction
# Green label = correct, Red label = incorrect
plot_random_image(
    model=model_7,
    images=test_data,
    true_labels=test_labels,
    classes=class_names
)